# 5 — Tabelas de Dados

In [ ]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import sys
import os

sys.path.append(os.path.abspath(".."))

from utils import (
    carregar_dados,
    calcular_indicadores,
    gerar_mapa_cores,
    aplicar_layout_light,
    escala_indicador,
    CORES_INDICADORES,
    CORES_CATEGORIA,
    CORES_CONCLUSAO_LIGHT,
    CORES_FLUXO_LIGHT,
    CORES_MATRICULAS_ATIVAS_LIGHT,
    CORES_SITUACAO_LIGHT,
    COR_TEMPO_MEDIANO,
    CORES_CURSO,
    CORES_SITUACAO,
    PALETA_QUALITATIVA_LIGHT,
    ROTULOS_INDICADORES,
)


df_completo = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")
print(df_completo.shape)
df_completo.head()


## Filtros

In [ ]:
periodo = (int(df_completo['Ano'].min()), int(df_completo['Ano'].max()))
tipos_selecionados = sorted(df_completo['Tipo de Curso'].dropna().unique())
cursos_selecionados = sorted(df_completo['Nome de Curso'].dropna().unique())

df = df_completo[
    (df_completo['Ano'] >= periodo[0])
    & (df_completo['Ano'] <= periodo[1])
    & (df_completo['Tipo de Curso'].isin(tipos_selecionados))
    & (df_completo['Nome de Curso'].isin(cursos_selecionados))
].copy()
print(df.shape)

## Indicadores por ano — campus completo

In [ ]:
ind_ano = calcular_indicadores(df, ['Ano'])
ind_ano

## Indicadores por ano e curso

In [ ]:
ind_ano_curso = calcular_indicadores(df, ['Ano', 'Nome de Curso'])
ind_ano_curso.head(20)

## Consolidado por curso no período

Recalcula numeradores e denominadores considerando todo o período filtrado.

In [ ]:
consolidado_curso = calcular_indicadores(df, ['Nome de Curso'])
consolidado_curso.sort_values('TE', ascending=False)

## Média anual por curso

Calcula os indicadores por ano e curso e depois tira a média dos percentuais anuais.

In [ ]:
media_anual_curso = (
    ind_ano_curso.groupby('Nome de Curso')[['TC', 'TE', 'TR', 'IEf', 'TPE']]
    .mean()
    .reset_index()
    .round(2)
)
media_anual_curso.sort_values('TE', ascending=False)

## Comparação metodológica: consolidado × média anual

In [ ]:
comparacao = consolidado_curso[['Nome de Curso', 'TC', 'TE', 'TR', 'IEf', 'TPE']].merge(
    media_anual_curso, on='Nome de Curso', suffixes=('_consolidado', '_media_anual')
)
comparacao

## Exportação opcional

In [ ]:
# Descomente para exportar as tabelas
# ind_ano.to_csv('indicadores_por_ano.csv', index=False, sep=';', decimal=',')
# ind_ano_curso.to_csv('indicadores_por_ano_curso.csv', index=False, sep=';', decimal=',')
# consolidado_curso.to_csv('indicadores_consolidado_curso.csv', index=False, sep=';', decimal=',')
# media_anual_curso.to_csv('indicadores_media_anual_curso.csv', index=False, sep=';', decimal=',')